<a href="https://colab.research.google.com/github/tiagopessoalima/POO/blob/main/Aula_16_(POO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Iteráveis e Iteradores**

Em Python, a maior parte das estruturas que manipulam coleções — como listas, strings e dicionários — funciona graças a um conjunto de regras chamado **protocolo de iteração**. Esse protocolo define como um objeto deve se comportar para que seus elementos possam ser percorridos
um por vez, seja em um `for`, em compreensões de listas, ou em funções como `sum()` e `min()`.

Para entender esse protocolo, precisamos distinguir dois conceitos fundamentais:

- **Iterável** → o objeto que *pode ser percorrido*
- **Iterador** → o objeto que *realiza a iteração de fato*, retornando o próximo elemento

Vamos explorar cada um deles.

## **Definições**

### **O que são Iteráveis?**


Um **iterável** é qualquer objeto que pode ser percorrido em um loop, ou seja, que pode retornar seus elementos um por vez. Ele segue o protocolo de iteração por meio do método `__iter__()`, que retorna um *iterador*.


#### **Características**

- Possui um método `__iter__()` que retorna um iterador
- Pode ser usado em loops *for*, *comprehensions*, etc.
- Exemplos: listas, tuplas, strings, dicionários, conjuntos

#### **Exemplo**

No `for`, *Python* automaticamente obtém um iterador do iterável:

In [ ]:
lista = [10, 20, 30]

for item in lista:   # lista é um iterável
    print(item)

10
20
30


### **O que são Iteradores?**

Um **iterador** é o objeto que implementa o protocolo de iteração. Ele sabe *qual é o próximo elemento* e quando a sequência terminou.

- Possui método `__iter__()` que retorna o próprio iterador
- Possui método `__next__()` que retorna o próximo elemento
- Levanta `StopIteration` quando não há mais elementos

#### **Como Obter um Iterador de um Iterável**?

Você cria um iterador chamando `iter()` sobre um iterável

In [8]:
lista = [10, 20, 30]
it = iter(lista)    # 'it' é um iterador

Agora você pode usar `next()`:

In [9]:
lista = [10, 20, 30]
it = iter(lista)

print(next(it))  # 10
print(next(it))  # 20
print(next(it))  # 30

# Este chamará StopIteration
# print(next(it))

10
20
30


## **Protocolo de Iteração**

Para um objeto ser considerado um Iterador válido em *Python*, ele deve satisfazer as seguintes interfaces:

| Método        | Assinatura               | Comportamento Técnico Esperado |
|---------------|---------------------------|---------------------------------|
| `__iter__()`  | `(self) -> self`          | Retorna o próprio objeto iterador, permitindo que iteradores sejam usados onde se espera iteráveis (polimorfismo). |
| `__next__()`  | `(self) -> Any`           | Retorna o próximo item do fluxo. Deve levantar a exceção `StopIteration` quando não houver mais dados. |


### **Observação**





Todo iterador é iterável (tem `__iter__()`), mas nem todo iterável é iterador (pode não ter `__next__()`).

- **Iterável que não é iterador**

In [ ]:
lista = [1, 2, 3]

# A lista é iterável
print(hasattr(lista, "__iter__"))   # True

# Mas a lista NÃO é um iterador (não tem __next__)
print(hasattr(lista, "__next__"))   # False

True
False


- **Iterador que também é iterável**

In [ ]:
lista = [1, 2, 3]
it = iter(lista)

# O iterador possui __iter__
print(hasattr(it, "__iter__"))   # True

# E possui __next__
print(hasattr(it, "__next__"))   # True

## **Como o `for` realmente funciona**

O *loop* `for` em *Python* é uma abstração sintática que esconde a complexidade do protocolo. Esta transformação ocorre:



**Abstração de Alto Nível:**

In [ ]:
dados = [10, 20, 30]
for elemento in dados:
    print(elemento)

10
20
30


**Implementação de Baixo Nível:**

In [ ]:
iterador = iter(dados)        # 1. Obtém iterador
while True:
    try:
        elemento = next(iterador)  # 2. Avança iteração
        print(elemento)            #    Executa corpo
    except StopIteration:          # 3. Trata término
        break

10
20
30


## **Exemplo Prático: Implementação Customizada**


In [ ]:
class ContagemRegressiva:
    """Iterável que conta de N até 1"""
    def __init__(self, inicio):
        self.inicio = inicio

    def __iter__(self):
        return ContagemRegressivaIterador(self.inicio)

class ContagemRegressivaIterador:
    """Iterador para contagem regressiva"""
    def __init__(self, valor):
        self.valor_atual = valor

    def __iter__(self):
        return self

    def __next__(self):
        if self.valor_atual < 1:
            raise StopIteration
        valor = self.valor_atual
        self.valor_atual -= 1
        return valor

# Uso
for numero in ContagemRegressiva(3):
    print(numero)  # 3, 2, 1

3
2
1


### **Classe ContagemRegressiva (Iterável)**

#### `__iter__(self)`

**Propósito:** Implementa o método obrigatório do protocolo de iteração para objetos iteráveis.

**Funcionamento:**
- É chamado automaticamente por `iter()` ou pelo loop `for`;
- Retorna uma nova instância do iterador (ContagemRegressivaIterador);
- Passa self.inicio como parâmetro para o iterador;
- Cada chamada cria um iterador independente.

**Fluxo quando usado em for:**

In [ ]:
# Quando Python executa:
for numero in ContagemRegressiva(3):
    print(numero)

# Internamente faz:
# 1. obj = ContagemRegressiva(3)
# 2. iterador = obj.__iter__()  → ContagemRegressivaIterador(3)
# 3. Loop usando next(iterador)

### **Classe ContagemRegressivaIterador (Iterador)**


#### `__iter__(self)`

**Propósito:** Implementa o método que faz do iterador também um iterável.

**Por que isso é necessário?**
1. **Protocolo exige:** Todo iterador deve ter `__iter__()` que retorna a si mesmo
2. **Polimorfismo:** Permite usar iteradores onde se espera iteráveis
3. **Compatibilidade:** Permite passar iteradores diretamente para *loops* `for`

**Exemplo de uso:**

In [ ]:
iterador = ContagemRegressivaIterador(3)

# Funciona porque iterador tem __iter__()
for x in iterador:  # iter(iterador) → iterador.__iter__() → self
    print(x)

3
2
1


#### `__next__(self)`

**Propósito:** Avança a iteração e retorna o próximo valor.



**Passo a passo da execução** (Estado inicial: self.valor_atual = 3):

- Primeira chamada:
```python
# self.valor_atual = 3
if 3 < 1:  # False
valor = 3  # Guarda o valor atual
self.valor_atual = 2  # Decrementa para próxima iteração
return 3  # Retorna 3
```

- Segunda chamada:
```python
# self.valor_atual = 2
if 2 < 1:  # False
valor = 2
self.valor_atual = 1
return 2
```

- Terceira chamada:
```python
# self.valor_atual = 1
if 1 < 1:  # False
valor = 1
self.valor_atual = 0
return 1
```

- Quarta chamada:
```python
# self.valor_atual = 0
if 0 < 1:  # TRUE
raise StopIteration  # Sinaliza fim da iteração
```

**Mecânica do StopIteration:**
- Não é um "erro" no sentido tradicional
- É um sinal de controle de fluxo
- O loop for captura esta exceção e entende que deve terminar
- Equivalente a break em um loop manual

#### **Por que Separar em Duas Classes?**

##### **Vantagens da Separação**

1. Múltiplas iterações independentes:

In [ ]:
contagem = ContagemRegressiva(3)

# Dois iteradores independentes
iter1 = iter(contagem)  # Novo ContagemRegressivaIterador(3)
iter2 = iter(contagem)  # Novo ContagemRegressivaIterador(3)

print(next(iter1))  # 3
print(next(iter2))  # 3 (não afetado por iter1)

3
3


2. Reutilização do iterável:

In [ ]:
contagem = ContagemRegressiva(3)

# Primeira passagem
for x in contagem:
    print(x)  # 3, 2, 1

# Segunda passagem (cria novo iterador)
for x in contagem:
    print(x)  # 3, 2, 1 (funciona novamente!)

3
2
1
3
2
1


3. Clareza conceitual:
- Iterável: Define o que será iterado
- Iterador: Define como iterar e mantém estado

#### **Alternativa (Não Recomendada):**

In [ ]:
# Misturando iterável e iterador (problema: só pode iterar uma vez)
class ContagemUnica:
    def __init__(self, inicio):
        self.valor_atual = inicio

    def __iter__(self):
        return self  # PROBLEMA: Retorna si mesmo

    def __next__(self):
        if self.valor_atual < 1:
            raise StopIteration
        valor = self.valor_atual
        self.valor_atual -= 1
        return valor

contagem = ContagemUnica(3)
print(list(contagem))  # [3, 2, 1]
print(list(contagem))  # [] - Já esgotado!

[3, 2, 1]
[]


## **Exercício**

Implementar uma **estrutura de dados encadeada** que respeite o **protocolo de iteração do Python**, decompondo o problema em:

1. **Representação da estrutura (nós e ponteiros)**
2. **Provedor de iteradores (iterable provider)**
3. **Iterador conforme a interface `Iterator`**

### **Especificação Técnica**

#### Estruturas que você deve implementar:

* `Node`
* `LinkedList` (iterável)
* `LinkedListIterator` (iterador conforme protocolo Python)

### 1. **Classe `Node` — Unidade atômica da estrutura**

Cada nó deve armazenar:

* o valor do usuário (`valor`)
* ponteiro para o próximo nó (`proximo`)

**Invariante estrutural:**
Se `n.proximo is None`, então `n` representa o último elemento da lista.

```python
class Node:
    def __init__(self, valor):
        self.valor = valor
        self.proximo = None
```

### 2. **Classe `LinkedList` — Estrutura encadeada e iterável**

A lista deve manter:

* Referência para o nó inicial (`head`)

#### **Requisitos funcionais**

##### 2.1. `append(self, valor)`

* Operação deve ter **custo O(n)** (varrer a lista até o último nó)
* `head` deve ser atualizado apenas se a lista estiver vazia
* O novo nó deve ser anexado ao final

##### 2.2. `__iter__(self)`

Implementar o **protocolo de iteração**, retornando um objeto que implemente:

* `__iter__()` → `self`
* `__next__()` → próximo valor da sequência

Ou seja, **LinkedList é um iterável**, mas **não é um iterador**.

```python
class LinkedList:
    def __init__(self):
        self.head = None
    
    def append(self, valor):
        # implementar
        pass
    
    def __iter__(self):
        # deve retornar LinkedListIterator(head)
        pass
```

### 3. **Classe `LinkedListIterator` — Estado e avanço da iteração**

#### Requisitos de implementação

##### 3.1. `__init__(self, node_atual)`

* Recebe um nó inicial
* Não deve alterar a estrutura subjacente

##### 3.2. `__iter__(self)`

* Deve retornar **o próprio iterador**
* Garante compatibilidade com o protocolo que exige que iteradores sejam também iteráveis

##### 3.3. `__next__(self)`

* Retorna o valor do nó atual
* Avança ponteiro interno (`self.atual = self.atual.proximo`)
* Quando `self.atual is None`, levantar:

```python
raise StopIteration
```

> **Obrigatório:** Não retornar `None` para sinalizar término — o protocolo exige `StopIteration`.

### **Implementação do Iterador (Esqueleto)**

```python
class LinkedListIterator:
    def __init__(self, node_atual):
        self.atual = node_atual
    
    def __iter__(self):
        return self
    
    def __next__(self):
        # implementar lógica de avanço
        pass
```

### **Comportamento Esperado (Contrato de Uso)**

```python
lista = LinkedList()
lista.append(10)
lista.append(20)
lista.append(30)

for valor in lista:
    print(valor)
```

**Saída obrigatória:**

```
10
20
30
```

#### Garantias que sua solução **deve** fornecer

* Cada chamada de `iter(lista)` deve produzir um **novo iterador independente**
* A lista não deve ser consumida após uma iteração
* O iterador não modifica a estrutura da lista
* O iterador avança em **tempo O(1)** por elemento

### **Análise de Complexidade**

| Operação                        | Complexidade |
| ------------------------------- | ------------ |
| `append(valor)`                 | O(n)         |
| Inicialização do iterador       | O(1)         |
| `next()` em cada elemento       | O(1)         |
| Iteração completa (n elementos) | O(n)         |